# Scrapping the DSE Winter Schools
**Note:** For reproducibility:



1.   First make a copy of the notebook and work on the copy
2.   Check these things: paths name, file names, APIs IDs, and customize them according to your environment
3.   After doing a major data operation, you can push your final version here: https://github.com/GaelKBertrand/scholar_info_append






### Mounting Google Drive as a working director

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Installing and loading all dependencies and libraries needed

In [ ]:
#Install necessary dependencies and libraries
!pip install waybackpy
!pip install googlesearch-python
!pip install google-api-python-client
!pip install beautifulsoup4
!pip install fake_useragent #necessary for google search rapid automation
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!apt-get install -y wget
!pip install requests-html

import requests
from bs4 import BeautifulSoup
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import logging


from waybackpy import WaybackMachineSaveAPI
from googlesearch import search
from googleapiclient.discovery import build
import time
import requests
from fake_useragent import UserAgent
from bs4 import BeautifulSoup
import logging

#from requests_html import HTMLSession
#from requests_html import AsyncHTMLSession
import pandas as pd
import logging
import asyncio
import nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.1/201.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.7/481.7 kB 25.8 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [2,741 kB]
Get:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu ja

# Cleaning the datasets

## Cleaning presentations

In [32]:
import pandas as pd

# Load the CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2023t.csv'
df = pd.read_csv(file_path)

# Step 3: Clean the 'Session' column by removing everything after "Session" number
df['Session'] = df['Session'].str.extract(r'(Session \d+)')




In [18]:
# Step 2: Check all column names
print("Column names in the dataset:")
print(df.columns)

# Step 3: Check if there is a last column without a name
if df.columns[-1] == '':
    # Step 4: Name that last column 'Misc'
    df.columns.values[-1] = 'Misc'
    print("Last column was unnamed, it has been renamed to 'Misc'.")
else:
    print("Last column is properly named.")


Column names in the dataset:
Index(['Session', 'Session topic', 'Presentation title', 'Author numbers',
       'Authors', 'Author Affiliation', 'Unnamed: 6', 'Unnamed: 7'],
      dtype='object')
Last column is properly named.


In [33]:
# Step 2: Check all column names
print("Column names in the dataset:")
print(df.columns)


Column names in the dataset:
Index(['Session', 'Session topic', 'Presentation title', 'Author numbers',
       'Authors', 'Author Affiliation', 'Unnamed: 6', 'Unnamed: 7'],
      dtype='object')


In [17]:
# Step 4: Process the 'Author numbers', 'Presentation title', 'Authors', 'Author Affiliation', and 'Misc' columns

for index, row in df.iterrows():
    # Step 4a: Handle 'Author numbers' if it's not a number or number as string
    if not row['Author numbers'].isdigit():
        # Add the text from 'Author numbers' to 'Presentation title'
        df.at[index, 'Presentation title'] = str(row['Presentation title']) + ' ' + str(row['Author numbers'])
        # Clear 'Author numbers'
        df.at[index, 'Author numbers'] = None

    # Step 4b: Check 'Authors' and adjust 'Author numbers' or 'Presentation title' accordingly
    if not str(row['Authors']).isdigit():
        # Add the text from 'Authors' to 'Presentation title'
        df.at[index, 'Presentation title'] = str(row['Presentation title']) + ' ' + str(row['Authors'])
        # Clear 'Authors'
        df.at[index, 'Authors'] = None
    else:
        # If 'Authors' contains a number, move it to 'Author numbers'
        df.at[index, 'Author numbers'] = row['Authors']
        df.at[index, 'Authors'] = None

    # Step 4c: Check 'Author Affiliation' and adjust 'Author numbers' or 'Authors' accordingly
    if not str(row['Author Affiliation']).isdigit():
        # Add the text from 'Author Affiliation' to 'Authors'
        df.at[index, 'Authors'] = str(row['Authors']) + ' ' + str(row['Author Affiliation'])
        # Clear 'Author Affiliation'
        df.at[index, 'Author Affiliation'] = None
    else:
        # If 'Author Affiliation' contains a number, move it to 'Author numbers'
        df.at[index, 'Author numbers'] = row['Author Affiliation']
        df.at[index, 'Author Affiliation'] = None

    # Step 4d: Move 'Misc' to 'Author Affiliation'
    #df.at[index, 'Author Affiliation'] = str(row['Author Affiliation']) + ' ' + str(row['Misc'])
    # Clear 'Misc'
    #df.at[index, 'Misc'] = None

# Save the cleaned dataframe back to CSV
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2021_cleaned.csv'
df.to_csv(output_file_path, index=False)

print("Cleaning is complete and saved to:", output_file_path)

KeyError: 'Misc'

In [8]:
import pandas as pd

# Load the CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2020t.csv'
df = pd.read_csv(file_path)

# Step 3: Clean the 'Session' column by removing everything after "Session" number
df['Session'] = df['Session'].str.extract(r'(Session \d+)')

# Step 4: Process the 'Author numbers', 'Presentation title', 'Authors', 'Author Affiliation', and 'Misc' columns

for index, row in df.iterrows():
    # Step 4a: Handle 'Author numbers' if it's not a number or number as string
    if not row['Author numbers'].isdigit():
        # Add the text from 'Author numbers' to 'Presentation title'
        df.at[index, 'Presentation title'] = str(row['Presentation title']) + ' ' + str(row['Author numbers'])
        # Remove 'Author numbers' content
        df.at[index, 'Author numbers'] = None

    # Step 4b: Check 'Authors' and adjust 'Author numbers' or 'Presentation title' accordingly
    if not str(row['Authors']).isdigit():
        # Add the text from 'Authors' to 'Presentation title'
        df.at[index, 'Presentation title'] = str(row['Presentation title']) + ' ' + str(row['Authors'])
        # Remove 'Authors' content
        df.at[index, 'Authors'] = None
    else:
        # If 'Authors' contains a number, move it to 'Author numbers'
        df.at[index, 'Author numbers'] = row['Authors']
        # Remove 'Authors' content
        df.at[index, 'Authors'] = None

    # Step 4c: Check 'Author Affiliation' and adjust 'Author numbers' or 'Authors' accordingly
    if not str(row['Author Affiliation']).isdigit():
        # Add the text from 'Author Affiliation' to 'Authors'
        df.at[index, 'Authors'] = str(row['Authors']) + ' ' + str(row['Author Affiliation'])
        # Remove 'Author Affiliation' content
        df.at[index, 'Author Affiliation'] = None
    else:
        # If 'Author Affiliation' contains a number, move it to 'Author numbers'
        df.at[index, 'Author numbers'] = row['Author Affiliation']
        # Remove 'Author Affiliation' content
        df.at[index, 'Author Affiliation'] = None

    # Step 4d: Move 'Misc' to 'Author Affiliation'
    df.at[index, 'Author Affiliation'] = str(row['Author Affiliation']) + ' ' + str(row['Misc'])
    # Remove 'Misc' content
    df.at[index, 'Misc'] = None

# Save the cleaned dataframe back to CSV
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned.csv'
df.to_csv(output_file_path, index=False)

print("Cleaning is complete and saved to:", output_file_path)


Cleaning is complete and saved to: /content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned.csv


In [9]:
import pandas as pd

# Load the CSV file (assuming you've already loaded the cleaned file)
file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned.csv'
df = pd.read_csv(file_path)

# Step 1: Remove any numbers at the beginning of names in the 'Authors' column
df['Authors'] = df['Authors'].str.replace(r'^\d+\s', '', regex=True)

# Save the cleaned dataframe back to CSV
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned_v2.csv'
df.to_csv(output_file_path, index=False)

print("Cleaning is complete and saved to:", output_file_path)


Cleaning is complete and saved to: /content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned_v2.csv


In [10]:
import pandas as pd

# Load the CSV file (assuming you're continuing with the cleaned file)
file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned_v2.csv'
df = pd.read_csv(file_path)

# Remove unwanted columns
df = df.drop(columns=['Misc', 'Unnamed: 7', 'Unnamed: 8'], errors='ignore')

# Save the cleaned dataframe back to CSV
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned_v3.csv'
df.to_csv(output_file_path, index=False)

print("Unwanted columns removed and saved to:", output_file_path)


Unwanted columns removed and saved to: /content/drive/My Drive/DSE/DSE_clean/presentations2020_cleaned_v3.csv


Adding countries

In [42]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import openpyxl

# Initialize geolocator with a rate limiter
geolocator = Nominatim(user_agent="geo_affiliation_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_country_from_affiliation(affiliation):
    """
    Get the country name from the affiliation string using geopy, ensuring results are in English.
    """
    try:
        if pd.isnull(affiliation) or affiliation.strip() == "":
            return None
        location = geocode(affiliation, timeout=10)
        if location and location.raw.get('display_name'):
            address_parts = location.raw['display_name'].split(",")
            # Extract and clean the country name
            country = address_parts[-1].strip()
            return country
        return None
    except Exception as e:
        return None

# Load your CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/presentation2023t.csv'
df = pd.read_csv(file_path)

# Step 1: Identify the relevant columns for Affiliation
affiliation_col = "Author Affiliation"  # Replace with your actual column name
location_col = "Affiliation Location"
india_col = "Affiliation Location India"

# Step 2: Check if the columns exist, and add if necessary
if location_col not in df.columns:
    df[location_col] = None

if india_col not in df.columns:
    df[india_col] = None

# Step 3: Use geopy to get country information and update the columns
tqdm.pandas(desc="Processing Affiliations")
df[location_col] = df[affiliation_col].progress_apply(get_country_from_affiliation)

# Step 4: Update 'Affiliation Location India' based on whether the country is India
df[india_col] = df[location_col].apply(lambda x: 1 if x == 'India' else 0)

# Step 5: Save the updated DataFrame to a new CSV
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2023_cleaned_v3.csv'
df.to_csv(output_file_path, index=False)

print(f"Processing complete. Updated file saved as {output_file_path}.")


Processing Affiliations: 100%|██████████| 190/190 [02:58<00:00,  1.06it/s]

Processing complete. Updated file saved as /content/drive/My Drive/DSE/DSE_clean/presentations2023_cleaned_v3.csv.


Adding genders

In [14]:
!pip install requests
!pip install gender-guesser
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.3/379.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 3.0 MB/s eta 0:00:00


In [43]:
import pandas as pd
import gender_guesser.detector as gender
from tqdm import tqdm

# Function to get gender using gender-guesser
def get_gender(name):
    if not isinstance(name, str):  # Check if the name is a string
        return 'unknown'

    first_name = name.split()[0]  # Get first name
    d = gender.Detector()
    gender_guess = d.get_gender(first_name)

    # Convert the gender guess to a more readable format
    if gender_guess in ['male', 'mostly_male']:
        return 'Male'
    elif gender_guess in ['female', 'mostly_female']:
        return 'Female'
    else:
        return 'unknown'

# Load your CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2023_cleaned_v3.csv'
df = pd.read_csv(file_path)


# Step 5: Add Gender column based on the 'Authors' column
gender_col = "Gender"  # New column to store gender information
if gender_col not in df.columns:
    df[gender_col] = None

# Step 6: Apply gender detection to the 'Authors' column (or any other relevant column)
tqdm.pandas(desc="Processing Gender")
df[gender_col] = df['Authors'].progress_apply(get_gender)

# Step 7: Save the updated DataFrame to a new CSV file
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/presentations2023_cleaned_v3.csv'
df.to_csv(output_file_path, index=False)

print(f"Processing complete. Updated file saved as {output_file_path}.")

Processing Gender: 100%|██████████| 190/190 [01:13<00:00,  2.58it/s]

Processing complete. Updated file saved as /content/drive/My Drive/DSE/DSE_clean/presentations2023_cleaned_v3.csv.


## Cleaning chairs of sections

Load datasets and check

In [60]:
import pandas as pd

# Load the CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/chairs2023.csv'
df = pd.read_csv(file_path)

In [61]:
# Step 2: Check all column names
print("Column names in the dataset:")
print(df.columns)


Column names in the dataset:
Index(['Session', 'Chair', 'Affiliation', 'Unnamed: 3'], dtype='object')


Adding countries

In [62]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm
import openpyxl

# Initialize geolocator with a rate limiter
geolocator = Nominatim(user_agent="geo_affiliation_locator")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

def get_country_from_affiliation(affiliation):
    """
    Get the country name from the affiliation string using geopy, ensuring results are in English.
    """
    try:
        if pd.isnull(affiliation) or affiliation.strip() == "":
            return None
        location = geocode(affiliation, timeout=10)
        if location and location.raw.get('display_name'):
            address_parts = location.raw['display_name'].split(",")
            # Extract and clean the country name
            country = address_parts[-1].strip()
            return country
        return None
    except Exception as e:
        return None

# Load your CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/chairs2023.csv'
df = pd.read_csv(file_path)

# Step 1: Identify the relevant columns for Affiliation
affiliation_col = "Affiliation"  # Replace with your actual column name
location_col = "Affiliation Location"
india_col = "Affiliation Location India"

# Step 2: Check if the columns exist, and add if necessary
if location_col not in df.columns:
    df[location_col] = None

if india_col not in df.columns:
    df[india_col] = None

# Step 3: Use geopy to get country information and update the columns
tqdm.pandas(desc="Processing Affiliations")
df[location_col] = df[affiliation_col].progress_apply(get_country_from_affiliation)

# Step 4: Update 'Affiliation Location India' based on whether the country is India
df[india_col] = df[location_col].apply(lambda x: 1 if x == 'India' else 0)

# Step 5: Save the updated DataFrame to a new CSV
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/chairs2023_cleaned_v3.csv'
df.to_csv(output_file_path, index=False)

print(f"Processing complete. Updated file saved as {output_file_path}.")


Processing Affiliations: 100%|██████████| 36/36 [00:34<00:00,  1.05it/s]

Processing complete. Updated file saved as /content/drive/My Drive/DSE/DSE_clean/chairs2023_cleaned_v3.csv.


Adding genders

In [63]:
import pandas as pd
import gender_guesser.detector as gender
from tqdm import tqdm

# Function to get gender using gender-guesser
def get_gender(name):
    if not isinstance(name, str):  # Check if the name is a string
        return 'unknown'

    first_name = name.split()[0]  # Get first name
    d = gender.Detector()
    gender_guess = d.get_gender(first_name)

    # Convert the gender guess to a more readable format
    if gender_guess in ['male', 'mostly_male']:
        return 'Male'
    elif gender_guess in ['female', 'mostly_female']:
        return 'Female'
    else:
        return 'unknown'

# Load your CSV file
file_path = '/content/drive/My Drive/DSE/DSE_clean/chairs2023_cleaned_v3.csv'
df = pd.read_csv(file_path)


# Step 5: Add Gender column based on the 'Authors' column
gender_col = "Gender"  # New column to store gender information
if gender_col not in df.columns:
    df[gender_col] = None

# Step 6: Apply gender detection to the 'Authors' column (or any other relevant column)
tqdm.pandas(desc="Processing Gender")
df[gender_col] = df['Chair'].progress_apply(get_gender)

# Step 7: Save the updated DataFrame to a new CSV file
output_file_path = '/content/drive/My Drive/DSE/DSE_clean/chairs2023_cleaned_v3.csv'
df.to_csv(output_file_path, index=False)

print(f"Processing complete. Updated file saved as {output_file_path}.")

Processing Gender: 100%|██████████| 36/36 [00:13<00:00,  2.77it/s]

Processing complete. Updated file saved as /content/drive/My Drive/DSE/DSE_clean/chairs2023_cleaned_v3.csv.


## Adding and merging them into one xlsl file

In [64]:
import pandas as pd

# List of CSV files for presentations and chairs
presentation_files = [
    '/content/drive/MyDrive/DSE/DSE_clean/presentations2020_cleaned_v3.csv',
    '/content/drive/MyDrive/DSE/DSE_clean/presentations2021_cleaned_v3.csv',
    '/content/drive/MyDrive/DSE/DSE_clean/presentations2022_cleaned_v3.csv',
    '/content/drive/MyDrive/DSE/DSE_clean/presentations2023_cleaned_v3.csv'
]

chair_files = [
    '/content/drive/MyDrive/DSE/DSE_clean/chairs2020_cleaned_v3.csv',
    '/content/drive/MyDrive/DSE/DSE_clean/chairs2021_cleaned_v3.csv',
    '/content/drive/MyDrive/DSE/DSE_clean/chairs2022_cleaned_v3.csv',
    '/content/drive/MyDrive/DSE/DSE_clean/chairs2023_cleaned_v3.csv'
]

# Function to load CSV files and convert to Excel with two sheets
def convert_to_excel(presentation_files, chair_files, year):
    # Load the presentation and chair data for the specific year
    presentations_df = pd.read_csv(presentation_files[year-2020])
    chairs_df = pd.read_csv(chair_files[year-2020])

    # Create an Excel file with two sheets: presentations and chairs
    output_file = f"/content/drive/MyDrive/DSE/DSE_clean/DSE{year}.xlsx"

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Write the DataFrames to different sheets
        presentations_df.to_excel(writer, sheet_name='presentations', index=False)
        chairs_df.to_excel(writer, sheet_name='chairs', index=False)

    print(f"Saved: {output_file}")

# Convert files for each year (2020-2023)
for year in range(2020, 2024):
    convert_to_excel(presentation_files, chair_files, year)

print("All files have been successfully converted.")


Saved: /content/drive/MyDrive/DSE/DSE_clean/DSE2020.xlsx
Saved: /content/drive/MyDrive/DSE/DSE_clean/DSE2021.xlsx
Saved: /content/drive/MyDrive/DSE/DSE_clean/DSE2022.xlsx
Saved: /content/drive/MyDrive/DSE/DSE_clean/DSE2023.xlsx
All files have been successfully converted.


# Adding Countries

Determine location in India or not

In [ ]:
import pandas as pd
import openpyxl

# Load Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2022.xlsx"  # Replace with the path to your file
xls = pd.ExcelFile(file_path)

# Get all sheets
all_sheets = {sheet_name: pd.read_excel(xls, sheet_name=sheet_name) for sheet_name in xls.sheet_names}

# Process all sheets
for sheet in all_sheets.keys():
    print(f"Processing sheet: {sheet}")
    df = all_sheets[sheet]

    # Check if the 'Affiliation Country' column exists
    if "Affiliation Country" not in df.columns:
        print(f"Error: 'Affiliation Country' column not found in {sheet}. Skipping.")
        continue

    # Create the new column 'Affiliation Location'
    df['Affiliation Location'] = df['Affiliation Country'].apply(lambda x: 1 if x == "India" else 0)

    # Update the sheet in the dictionary
    all_sheets[sheet] = df

# Save back to Excel, preserving column widths
output_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2019.xlsx"
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, sheet_df in all_sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        # Retain column widths
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        for col_num, column_cells in enumerate(sheet_df.columns, start=1):
            max_length = max(sheet_df[column_cells].astype(str).map(len).max(), len(column_cells))
            worksheet.column_dimensions[openpyxl.utils.get_column_letter(col_num)].width = max_length

print(f"Processing complete. Updated file with 'Affiliation Location' column saved as {output_path}.")


Processing sheet: Authorwise (2019)
Processing sheet: Chair of regular sessions
Processing sheet: SPECIAL sessions Keynote, Plena
Processing complete. Updated file with 'Affiliation Location' column saved as /content/drive/MyDrive/ISI/ISIclean/ISI2019.xlsx.


Add genders

In [ ]:
!pip install requests
!pip install gender-guesser
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.3/379.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 10.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import gender_guesser.detector as gender
from tqdm import tqdm

# Function to get gender using gender-guesser
def get_gender(name):
    if not isinstance(name, str):  # Check if the name is a string
        return 'unknown'

    first_name = name.split()[0]  # Get first name
    d = gender.Detector()
    gender_guess = d.get_gender(first_name)

    # Convert the gender guess to a more readable format
    if gender_guess in ['male', 'mostly_male']:
        return 'Male'
    elif gender_guess in ['female', 'mostly_female']:
        return 'Female'
    else:
        return 'unknown'

# Load the Excel file
file_path = "/content/drive/MyDrive/ISI/ISIclean/ISI2019.xlsx"  # Replace with your actual file path
xls = pd.ExcelFile(file_path)

# List of sheets and the corresponding column names to process
sheet_columns = {
    'Authorwise (2019)': 'Author',
    'SPECIAL sessions Keynote, Plena': 'Name of chair',
    'Chair of regular sessions': 'Presenter/Panelist/Moderator'
}

# Iterate through each sheet and process it
with pd.ExcelWriter("processed_ISI.xlsx", engine='openpyxl') as writer:
    for sheet_name in xls.sheet_names:
        df = pd.read_excel(xls, sheet_name=sheet_name)

        # Check if the column exists in the current sheet
        if sheet_name in sheet_columns and sheet_columns[sheet_name] in df.columns:
            column_name = sheet_columns[sheet_name]

            # Use tqdm for a progress bar while processing rows
            tqdm.pandas(desc=f"Processing {sheet_name}")

            # Apply gender detection for the specified column
            df['Gender'] = df[column_name].progress_apply(get_gender)

        # Write the DataFrame back to the Excel writer
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Processing completed!")


Processing Authorwise (2019): 100%|██████████| 369/369 [02:11<00:00,  2.81it/s]

Processing completed!


In [ ]:
import spacy
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Function to process text with NLP and add markers
def add_markers_with_nlp(text):
    doc = nlp(text)
    updated_text = []
    delimiter = "***NEW PAPER***"

    session_re = re.compile(r'(Session \d+[a-z]* [^\n]* \([^\)]*\)\n)')
    session_blocks = session_re.split(text)

    for i, block in enumerate(session_blocks):
        if i % 2 == 0:
            updated_text.append(block)
        else:
            papers = re.split(r'(?:"[^"]+"\s*\n)', block)  # Split block at paper titles
            for j, paper in enumerate(papers):
                if j == 0:
                    updated_text.append(paper)
                else:
                    updated_text.append(f'{delimiter} "{papers[j-1].split(": ")[-1].strip()}" {paper}')

    return ''.join(updated_text)

# Assuming 'text' is already loaded
updated_text = add_markers_with_nlp(text)

# Print the updated text
print(updated_text)


Parallel Sessions 1                                                18 December, Monday 11:50 AM- 01:00  PM    
Session 1(a) Social Identity and Financial Inclusion (Conference Room) 
Rikhia Bhukta (Indian Institute of Technology Kanpur), Debayan Pakrashi (Indian Statistical Institute Kolkata; 
Indian Institute of Technology Kanpur), Sarani Saha (Indian Institute of Technology Kanpur), and Ashish Sedai 
(University of Texas at Arlington): "Does Financial Inclusion Mitigate Social Exclusion? Causal Evidence from 
India" 

Sandhya Garg (Institute of Economic Growth), Samarth Gupta (Ahmedabad University), and Sushanta Mallick 
(Queen Mary University of London): "Does Social Identity Constrain Rural Entrepreneurship? The Role of 
Financial Inclusion" 
Session 1(b) Welfare Programmes I (Seminar Room 1) 
Vidhya Unnikrishnan (University of Manchester): "Impact of Social Protection Policy Targeted on the 
Disabled: Panel Data Evidence from India" 


Michael Palmer (University of Western Austral

In [ ]:
import spacy
import pandas as pd
import re

# Load the English NLP model
nlp = spacy.load("en_core_web_sm")

# Define a function to extract session details using NLP
def extract_session_details(text):
    doc = nlp(text)

    sessions = []

    # Regular expression to identify sessions and topics
    session_re = re.compile(r"Session (\d+[a-z]*)([^\n]*)\s*\(([^)]*)\)")

    # Regular expression to match the authors and affiliations
    author_re = re.compile(r"([A-Za-z\s]+)\s?\(([^)]+)\)")

    # Regular expression to capture paper titles
    title_re = re.compile(r'"\*\*\*NEW PAPER\*\*\* ([^\"]+)"')

    # Iterate over all sessions in the text
    for match in session_re.finditer(text):
        session_number, session_topic, location = match.groups()

        # Locate the paper(s) associated with the session
        session_start = match.end()
        session_end = text.find("Session", session_start)  # Find next session to stop the current one
        session_content = text[session_start:session_end] if session_end != -1 else text[session_start:]

        # Extract the paper titles
        paper_titles = title_re.findall(session_content)

        # Extract authors and their affiliations
        authors_and_affiliations = author_re.findall(session_content)

        # Process each paper
        for paper_title in paper_titles:
            authors = []
            affiliations = []

            for author_name, affiliation in authors_and_affiliations:
                authors.append(author_name.strip())
                affiliations.append(affiliation.strip())

            # Loop over authors to create a row for each
            for author, affiliation in zip(authors, affiliations):
                sessions.append({
                    "Session": session_number,
                    "Topic": session_topic,
                    "Title of Paper": paper_title,
                    "Number of Authors": len(authors),
                    "Author": author,
                    "Institute/Affiliation": affiliation
                })

    # Convert the session details into a DataFrame
    df = pd.DataFrame(sessions)

    return df

# Extract session details from the conference text
conference_df = extract_session_details(updated_text)

# Display the structured DataFrame
print(conference_df)

# Save to CSV
conference_df.to_csv("/content/drive/My Drive/conference_details.csv", index=False)


Empty DataFrame
Columns: []
Index: []
